In [1]:
import sys
import pandas as pd
import numpy as np

# Printing the versions
print(f"Python version:  {sys.version}")
print(f"Pandas version:  {pd.__version__}")
print(f"NumPy version:   {np.__version__}")

Python version:  3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Pandas version:  3.0.3
NumPy version:   2.4.6


In [2]:
df = pd.read_csv('data/indian_employee_data.csv', encoding='utf-8')
# Displaying the first 10 records to inspect the structure
display(df.head(10))

,Emp_ID,Name,Age,Salary (INR),Experience (Years),City,Department,Performance Rating
0,101,Priya Tiwari,37,1029000.0,14.0,Jaipur,HR,4.4
1,102,Vinod Chopra,46,1506000.0,25.0,Indore,Marketing,4.0
2,103,Ramesh Yadav,46,1551000.0,24.0,Noida,Marketing,3.9
3,104,Karan Rao,38,1003000.0,14.0,Bhopal,HR,3.9
4,105,Riya Agrawal,35,902000.0,13.0,Indore,Operations,3.2
5,106,Vikram Yadav,29,NaN,6.0,Bangalore,IT,4.3
6,107,Ravi Chauhan,26,642000.0,3.0,Ahmedabad,Marketing,4.8
7,108,Isha Kapoor,32,990000.0,11.0,Bangalore,Sales,3.4
8,109,Alka Bhatt,29,869000.0,6.0,Pune,Marketing,3.5
9,110,Kavita Chopra,37,1477000.0,15.0,Surat,Management,4.0


In [3]:
print(f'Dataset Dimensions: {df.shape}')
print(f'Missing values per column: \n{df.isnull().sum()}')

Dataset Dimensions: (1001, 8)
Missing values per column: 
Emp_ID                 0
Name                   0
Age                    0
Salary (INR)          40
Experience (Years)    40
City                  40
Department             0
Performance Rating    40
dtype: int64


In [4]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

print(f"Duplicate rows found: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)

print(f"Data Shape after removing duplicates: {df.shape}")

Duplicate rows found: 1
Data Shape after removing duplicates: (1000, 8)


In [5]:
# find impossible negative salary records
negative_count = (df['Salary (INR)'] <0).sum()
print(f"Negative Salary entries detected: {negative_count}")

# Convert negative values to NaN
df.loc[df['Salary (INR)']<0, 'Salary (INR)'] = np.nan

Negative Salary entries detected: 14


## Strategic Missing Value Imputation
Instead of dropping rows with empty values, we fill them using basic statistical rules
* **Salary:** Filled with arithmetic **Mean** (Average)
* **Experience:** Filled with the **Median** (Middle value) to protect against skew from high-tenure employees.
* **Performance Rating:** Converted safely to numeric first, then filled with its **Median**.
* **City:** Filled with the **Mode** (Most common city name).

In [6]:
# 1. Fill Salary using the clean average
salary_mean = df['Salary (INR)'].mean()
df['Salary (INR)'] = df['Salary (INR)'].fillna(salary_mean)

# 2. Fill Experience using the median
experience_median = df['Experience (Years)'].median()
df['Experience (Years)'] = df['Experience (Years)'].fillna(experience_median)

# 3. Handle Performance Rating (force bad values to NaN then fill the median)
df['Performance Rating'] = pd.to_numeric(df['Performance Rating'], errors='coerce')
rating_median = df['Performance Rating'].median()
df['Performance Rating'] = df['Performance Rating'].fillna(rating_median)

# 4. Fill missing cities using the most frequent occurring city
city_mode = df['City'].mode()[0]
df['City'] = df['City'].fillna(city_mode)

print("All missing value gaps filled and saved successfully!")

All missing value gaps filled and saved successfully!


In [7]:
# Outlier Boundary Mapping
salary_mean = df['Salary (INR)'].mean()
salary_std = df['Salary (INR)'].std()

# Formulate standard boundaries
lower_bound = salary_mean - (3 * salary_std)
upper_bound = salary_mean + (3 * salary_std)

print(f"Calculated Mean:        {salary_mean:.2f} INR")
print(f"Calculated Std Dev:     {salary_std:.2f} INR")
print(f"Valid Salary Range:     [{lower_bound:.2f} to {upper_bound:.2f}]")

# Count outliers
outliers = df[(df['Salary (INR)'] < lower_bound) | (df['Salary (INR)'] > upper_bound)]
print(f"Number of outlier rows detected: {len(outliers)}")

Calculated Mean:        1098516.20 INR
Calculated Std Dev:     282382.04 INR
Valid Salary Range:     [251370.07 to 1945662.33]
Number of outlier rows detected: 1


In [8]:
print("Remaining Missing Values check: ")
print(df.isnull().sum())

print("\nPipeline Complete! Final Dataset Shape: ",df.shape)

pd.options.display.float_format = '{:.1f}'.format
display(df.head(10))

Remaining Missing Values check: 
Emp_ID                0
Name                  0
Age                   0
Salary (INR)          0
Experience (Years)    0
City                  0
Department            0
Performance Rating    0
dtype: int64

Pipeline Complete! Final Dataset Shape:  (1000, 8)


,Emp_ID,Name,Age,Salary (INR),Experience (Years),City,Department,Performance Rating
0,101,Priya Tiwari,37,1029000.0,14.0,Jaipur,HR,4.4
1,102,Vinod Chopra,46,1506000.0,25.0,Indore,Marketing,4.0
2,103,Ramesh Yadav,46,1551000.0,24.0,Noida,Marketing,3.9
3,104,Karan Rao,38,1003000.0,14.0,Bhopal,HR,3.9
4,105,Riya Agrawal,35,902000.0,13.0,Indore,Operations,3.2
5,106,Vikram Yadav,29,1098516.2,6.0,Bangalore,IT,4.3
6,107,Ravi Chauhan,26,642000.0,3.0,Ahmedabad,Marketing,4.8
7,108,Isha Kapoor,32,990000.0,11.0,Bangalore,Sales,3.4
8,109,Alka Bhatt,29,869000.0,6.0,Pune,Marketing,3.5
9,110,Kavita Chopra,37,1477000.0,15.0,Surat,Management,4.0


In [9]:
# Saving the cleaned dataframe to a csv
df.to_csv('data/cleaned_indian_employee_data.csv',index=False)

print("Data Cleaning completed! Saved as cleaned_indian_employee_data.csv")

Data Cleaning completed! Saved as cleaned_indian_employee_data.csv
